# Data Cleaning

## Objective

This notebook performs data cleaning on the raw datasets by handling missing values,
removing duplicates, correcting data types, validating records, and saving cleaned
datasets for analysis.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw" / "Bluestock_MF_Datasets"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(exist_ok=True)

## Cleaning NAV History Dataset

In [2]:
nav = pd.read_csv(
    RAW_DIR / "02_nav_history.csv"
)

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(
    by=["amfi_code", "date"]
)

nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)

nav = nav.drop_duplicates()

nav = nav[nav["nav"] > 0]

nav.to_csv(
    PROCESSED_DIR / "nav_history_clean.csv",
    index=False
)

print(nav.shape)
print("✅ NAV History Cleaned Successfully!")

(46000, 3)
✅ NAV History Cleaned Successfully!


## Cleaning Investor Transactions

In [5]:
transactions = pd.read_csv(
    RAW_DIR / "08_investor_transactions.csv"
)

transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [6]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)

transactions = transactions[
    transactions["amount_inr"] > 0
]

transactions["kyc_status"] = (
    transactions["kyc_status"]
    .str.strip()
    .str.title()
)

transactions = transactions.drop_duplicates()

transactions.to_csv(
    PROCESSED_DIR / "investor_transactions_clean.csv",
    index=False
)

print("✅ Transactions Cleaned Successfully!")

✅ Transactions Cleaned Successfully!


## Cleaning Scheme Performance Dataset

In [7]:
performance = pd.read_csv(
    RAW_DIR / "07_scheme_performance.csv"
)

performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [8]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

for col in return_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

performance.to_csv(
    PROCESSED_DIR / "scheme_performance_clean.csv",
    index=False
)

print("✅ Scheme Performance Cleaned Successfully!")

✅ Scheme Performance Cleaned Successfully!


## Data Cleaning Summary

### Completed Tasks

- Converted date columns to proper datetime format.
- Forward-filled missing NAV values.
- Removed duplicate records.
- Filtered invalid NAV and transaction amounts.
- Standardized transaction types and KYC status.
- Converted return columns to numeric format.
- Saved cleaned datasets to the `data/processed` folder.